# 05 — Generar ventanas sintéticas

Genera **5000** ventanas `[65, 3]` desde un checkpoint entrenado.
Salida: `outputs/synthetic_seed{SEED}_n5000.parquet`

Usar después de entrenar (notebook 02 o script `train_wgan_gp.py`).

In [7]:
from pathlib import Path
import numpy as np
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import tensorflow as tf
import yaml

from src.data import load_normalizer, synthetic_windows_to_frame
from src.io import load_generator, save_synthetic_parquet
from src.paths import artifacts_dir, outputs_dir

In [ ]:
SEED = 42
RUN_NAME = f"seed_{SEED}"          # o seed_42_notebook si entrenaste en nb 02
N_SYNTHETIC = 500                   # default configs/wgan_gp.yaml

run_dir = artifacts_dir() / RUN_NAME
checkpoints = sorted((run_dir / "checkpoints").glob("generator_epoch_*.keras"))
if not checkpoints:
    raise FileNotFoundError(f"No hay checkpoints en {run_dir / 'checkpoints'}")
checkpoint = checkpoints[-1]
normalizer = load_normalizer(run_dir / "normalizer.json")

print(f"Checkpoint: {checkpoint.name}")
print(f"Ventanas a generar: {N_SYNTHETIC}")

Checkpoint: generator_epoch_00002.keras
Ventanas a generar: 5000


In [9]:
generator = load_generator(checkpoint)
tf.keras.utils.set_random_seed(SEED)

noise = tf.random.normal([N_SYNTHETIC, generator.input_shape[-1]])
generated_norm = generator(noise, training=False).numpy()
generated = normalizer.denormalize(generated_norm.astype(np.float64))

output_path = outputs_dir() / f"synthetic_seed{SEED}_n{N_SYNTHETIC}.parquet"
frame = synthetic_windows_to_frame(
    generated,
    seed=SEED,
    ratio=None,
    checkpoint=str(checkpoint),
)
save_synthetic_parquet(frame, output_path)

print(f"Generadas {len(frame)} ventanas -> {output_path}")

Generadas 5000 ventanas -> /home/cristian/Documents/miax/modulo5/2026_Modelos_generativos/taller/taller_cristian/generadores/cristian/outputs/synthetic_seed42_n5000.parquet
